# MLOps II - Clase 1: Flujo de Datos para ML (RDBMS vs. Grafos)

**Objetivo de la práctica:** En el contexto de Machine Learning Ops, la latencia en la extracción de características (Feature Engineering) es crítica, especialmente para inferencia en tiempo real. 
En esta notebook compararemos el rendimiento y la complejidad de extraer características para un **Sistema de Recomendación** utilizando una Base de Datos Relacional (RDBMS) frente a una Base de Datos Orientada a Grafos (Neo4j) usando el dataset clásico **Northwind**.

## 1. Preparación del Entorno

Antes de comenzar, asegúrate de tener un contenedor de Neo4j corriendo en tu máquina y de instalar las librerías necesarias.

**Paso 1:** Ejecuta este comando en tu terminal para levantar Neo4j:
```bash
docker run --name neo4j-mlops -p7474:7474 -p7687:7687 -d -e NEO4J_AUTH=neo4j/password neo4j:latest
```
**Paso 2:** Ingresa a `http://localhost:7474` en tu navegador, inicia sesión (user: `neo4j`, pass: `password`) y ejecuta el comando `:play northwind-graph`. Importante!! Sigue los pasos interactivos para cargar los datos en tu grafo local.

In [ ]:
# Instalación de las librerías necesarias para la conexión y manipulación de datos
!pip install neo4j pandas

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for sqlite3


In [2]:
import pandas as pd
import sqlite3
import time
from neo4j import GraphDatabase

print("Librerías importadas correctamente. Listos para comenzar.")

Librerías importadas correctamente. Listos para comenzar.


## 2. El Enfoque Relacional (RDBMS)

Para simular el entorno relacional, crearemos una base de datos en memoria con `sqlite3` y cargaremos un subconjunto representativo de datos. 

En un sistema RDBMS, los datos están divididos en múltiples tablas. Para saber qué compró un cliente y qué más compraron otros clientes, debemos unir (JOIN) estas tablas.

In [3]:
# Configuramos una base de datos SQLite en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Creamos las tablas necesarias (Esquema Relacional)
cursor.executescript('''
    CREATE TABLE Customers (CustomerID TEXT PRIMARY KEY, CompanyName TEXT);
    CREATE TABLE Orders (OrderID INTEGER PRIMARY KEY, CustomerID TEXT);
    CREATE TABLE OrderDetails (OrderID INTEGER, ProductID INTEGER);
    CREATE TABLE Products (ProductID INTEGER PRIMARY KEY, ProductName TEXT);
''')

# Insertamos datos de prueba (Simulando un caso donde ALFKI compró Chai)
cursor.executescript('''
    INSERT INTO Customers VALUES ('ALFKI', 'Alfreds Futterkiste'), ('ANATR', 'Ana Trujillo'), ('BERGS', 'Berglunds');
    INSERT INTO Products VALUES (1, 'Chai'), (2, 'Chang'), (3, 'Aniseed Syrup');
    INSERT INTO Orders VALUES (1001, 'ALFKI'), (1002, 'ANATR'), (1003, 'BERGS');
    INSERT INTO OrderDetails VALUES (1001, 1), (1002, 1), (1002, 2), (1003, 1), (1003, 3);
''')
conn.commit()
print("Base de datos relacional simulada y poblada.")

Base de datos relacional simulada y poblada.


### 2.1. Feature Engineering RDBMS: Filtro Colaborativo
El objetivo es crear una *feature* para nuestro modelo: **"Recomendar productos basados en lo que compraron otros clientes con gustos similares"**.

Observa la cantidad de `JOINs` necesarios en SQL. Cada JOIN aumenta el costo computacional.

In [4]:
query_sql = """
SELECT DISTINCT p_recom.ProductName AS Recomendacion, COUNT(p_recom.ProductID) as Frecuencia
FROM Customers c
-- 1. Buscamos las órdenes del cliente objetivo ('ALFKI')
JOIN Orders o1 ON c.CustomerID = o1.CustomerID
JOIN OrderDetails od1 ON o1.OrderID = od1.OrderID
JOIN Products p_target ON od1.ProductID = p_target.ProductID
-- 2. Buscamos otros clientes que compraron ese mismo producto
JOIN OrderDetails od2 ON p_target.ProductID = od2.ProductID
JOIN Orders o2 ON od2.OrderID = o2.OrderID
JOIN Customers c_other ON o2.CustomerID = c_other.CustomerID
-- 3. Buscamos qué OTRAS cosas compraron esos clientes
JOIN Orders o3 ON c_other.CustomerID = o3.CustomerID
JOIN OrderDetails od3 ON o3.OrderID = od3.OrderID
JOIN Products p_recom ON od3.ProductID = p_recom.ProductID
WHERE c.CustomerID = 'ALFKI' 
  AND c_other.CustomerID <> 'ALFKI'
  AND p_recom.ProductID <> p_target.ProductID
GROUP BY p_recom.ProductName
ORDER BY Frecuencia DESC;
"""

start_time = time.time()
df_sql = pd.read_sql_query(query_sql, conn)
sql_duration = time.time() - start_time

print(f"Tiempo de ejecución SQL: {sql_duration:.5f} segundos")
display(df_sql)

Tiempo de ejecución SQL: 0.04960 segundos


,Recomendacion,Frecuencia
0,Chang,1
1,Aniseed Syrup,1


---
## 3. El Enfoque de Grafos (Neo4j)

En MLOps, muchas veces servimos predicciones mediante APIs REST. Si nuestra API necesita ejecutar 8 JOINs por cada request para extraer features, la base de datos colapsará. 

En un modelo de grafos, las "Foreign Keys" no existen como concepto lógico a resolver en tiempo de consulta; existen como **punteros físicos en memoria**.

Vamos a conectar nuestra notebook a Neo4j y ejecutar la misma lógica de recomendación.

In [7]:
# Conexión a Neo4j local
uri = "neo4j://localhost:7687"
user = "neo4j"
password = "password"

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_cypher_query(query):
    with driver.session() as session:
        result = session.run(query)
        return pd.DataFrame([r.values() for r in result], columns=result.keys())

print("Conexión a Neo4j establecida.")

Conexión a Neo4j establecida.


### 3.1. Feature Engineering Grafo: Filtro Colaborativo
En el lenguaje **Cypher**, dibujamos el patrón que estamos buscando usando arte ASCII. No hay JOINs. Solo decimos: 
`Cliente Objetivo -> compró -> Producto <- compró <- Otro Cliente -> compró -> Recomendación`

In [8]:
query_cypher = """
// 1. Buscamos las órdenes del cliente 'ALFKI' y los productos que compró
MATCH (c:Customer {customerID: 'ALFKI'})-[:PURCHASED]->(o1:Order)-[:ORDERS]->(p:Product)

// 2. Buscamos otros clientes que hayan comprado en otras órdenes esos mismos productos
MATCH (p)<-[:ORDERS]-(o2:Order)<-[:PURCHASED]-(other:Customer)
WHERE other <> c

// 3. Buscamos qué OTRAS cosas compraron esos clientes en cualquiera de sus órdenes
MATCH (other)-[:PURCHASED]->(o3:Order)-[:ORDERS]->(recom:Product)

// 4. Filtramos para no recomendar productos que 'ALFKI' ya haya comprado antes
WHERE NOT EXISTS {
  MATCH (c)-[:PURCHASED]->(:Order)-[:ORDERS]->(recom)
}

// 5. Devolvemos el resultado agregado
RETURN recom.productName AS Recomendacion, count(recom) AS Frecuencia
ORDER BY Frecuencia DESC
LIMIT 5
"""

start_time = time.time()
df_cypher = run_cypher_query(query_cypher)
cypher_duration = time.time() - start_time

print(f"Tiempo de ejecución Cypher: {cypher_duration:.5f} segundos")
display(df_cypher)

Tiempo de ejecución Cypher: 0.60993 segundos


,Recomendacion,Frecuencia
0,Gorgonzola Telino,368
1,Guaraná Fantástica,342
2,Camembert Pierrot,334
3,Chang,333
4,Jack's New England Clam Chowder,323


## 4. Conclusiones para MLOps

### Discusión en clase:
1. **Complejidad del Código:** ¿Cuál consulta es más fácil de mantener y evolucionar por un equipo de Data Engineering o Data Science?
2. **Desempeño a Escala (O(1) vs O(N^x)):** A medida que la base de datos SQL crece a millones de registros, los índices y JOINs se vuelven exponenciales. En Neo4j, el motor solo recorre el subgrafo local del cliente objetivo, manteniendo tiempos de respuesta constantes, ideales para **Feature Stores** y APIs en tiempo real.
3. **Flujo de Datos (Data Flow):** Considera cómo enviarías estos datos desde Neo4j hacia tu modelo. 